# Class 11 — classes in practice: solutions

This notebook contains solutions to the exercises of Class 11 (class methods, properties, when to use a class, inheritance and composition). It is self-contained: it starts from the classes `Tray` and `Inventory` of Class 9, and changes them step by step, as in the class. Each time a class changes, the whole class is written again, so that each cell can be read on its own.

Run the cells in order: later cells use the classes defined in earlier cells.

The solution of the *Cookie jar* assignment (5.5) is not included.

---
# Starting point: the classes of Class 9

Class `Tray` of Class 9 (3.1 and 3.2), and class `Inventory` with the methods of Class 9, Part 4 (4.1 to 4.6).

In [ ]:
class Tray:
    MAX_PLANTS = 50   # class attribute: shared by all trays

    def __init__(self, species, n_plants=0):
        if n_plants < 0 or n_plants > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self.species = species
        self.n_plants = n_plants

    def add_plants(self, n):
        if self.n_plants + n > Tray.MAX_PLANTS:
            raise ValueError(f"there is only room for {Tray.MAX_PLANTS - self.n_plants} more plants")
        self.n_plants += n

    def remove_plants(self, n):
        if n > self.n_plants:
            raise ValueError(f"the tray only has {self.n_plants} plants")
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"


class Inventory:
    def __init__(self, name):
        self.name = name
        self.trays = {}   # tray name -> Tray object

    def add_tray(self, tray_name, tray):
        self.trays[tray_name] = tray

    def get_tray(self, tray_name):
        return self.trays[tray_name]

    def remove_tray(self, tray_name):
        if tray_name not in self.trays:
            raise ValueError(f"there is no tray {tray_name}")
        del self.trays[tray_name]

    def list_all_trays(self):
        for tray_name, tray in self.trays.items():
            print(f"{tray_name}: {tray}")

    def n_plants(self):
        return sum(tray.n_plants for tray in self.trays.values())

    def plants_by_species(self):
        counts = {}
        for tray in self.trays.values():
            counts[tray.species] = counts.get(tray.species, 0) + tray.n_plants
        return counts

    def empty_trays(self):
        return [tray_name for tray_name, tray in self.trays.items() if tray.n_plants == 0]

    def total_value(self, prices):
        return sum(tray.n_plants * prices[tray.species] for tray in self.trays.values())

    def __str__(self):
        return f"{self.name}: {len(self.trays)} trays"

---
# Part 1 — Revision, and class methods

## Exercise 1.1

`n_trays` is a class variable: there is a single counter, shared by all trays, stored in the class. `__init__` increases it each time a tray is created.

The line `Tray.n_trays += 1` is at the **end** of `__init__`, after the validation: if `n_plants` is not valid, `raise` stops `__init__` before the counter is increased, so trays that could not be created are not counted.

In [ ]:
class Tray:
    MAX_PLANTS = 50   # class variable: the same for all trays
    n_trays = 0       # class variable: number of trays created

    def __init__(self, species, n_plants=0):
        if n_plants < 0 or n_plants > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self.species = species
        self.n_plants = n_plants
        Tray.n_trays += 1   # at the end: only trays that were created are counted

    def add_plants(self, n):
        if self.n_plants + n > Tray.MAX_PLANTS:
            raise ValueError(f"there is only room for {Tray.MAX_PLANTS - self.n_plants} more plants")
        self.n_plants += n

    def remove_plants(self, n):
        if n > self.n_plants:
            raise ValueError(f"the tray only has {self.n_plants} plants")
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"

In [ ]:
tray1 = Tray("Pansy", 20)
tray2 = Tray("Petunia", 30)
tray3 = Tray("Begonia")
print(Tray.n_trays)          # 3

try:
    Tray("Pansy", 60)        # not valid: not counted
except ValueError as e:
    print("ValueError:", e)
print(Tray.n_trays)          # still 3

Note that the counter must be increased with `Tray.n_trays += 1`, and not with `self.n_trays += 1`: assigning to `self.n_trays` would create a new attribute of **each** tray, equal to 1, and the counter of the class would stay at 0.

## 1.2 Class methods

`from_csv_row` receives the class as `cls`, and `cls(...)` does the same as `Tray(...)`. It is called on the class: `Tray.from_csv_row(...)`.

In [ ]:
class Tray:
    MAX_PLANTS = 50

    def __init__(self, species, n_plants=0):
        if n_plants < 0 or n_plants > Tray.MAX_PLANTS:
            raise ValueError(f"n_plants must be between 0 and {Tray.MAX_PLANTS}")
        self.species = species
        self.n_plants = n_plants

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants = row.split(",")
        return cls(species.strip(), int(n_plants))   # cls(...) is Tray(...)

    def add_plants(self, n):
        if self.n_plants + n > Tray.MAX_PLANTS:
            raise ValueError(f"there is only room for {Tray.MAX_PLANTS - self.n_plants} more plants")
        self.n_plants += n

    def remove_plants(self, n):
        if n > self.n_plants:
            raise ValueError(f"the tray only has {self.n_plants} plants")
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants"


tray = Tray.from_csv_row("Petunia,30")
print(tray)

---
# Part 2 — Properties

## 2.1 Trays of different sizes

The class of section 2.1, with `from_csv_row` of section 1.2:

In [ ]:
class Tray:
    CAPACITIES = (24, 50, 104)          # class variable: standard tray sizes

    def __init__(self, species, n_plants=0, capacity=50):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity       # must come before n_plants (see the setter)
        self.species = species
        self.n_plants = n_plants        # calls the setter below

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants = row.split(",")
        return cls(species.strip(), int(n_plants))

    @property
    def capacity(self):                 # read-only: there is no setter
        return self._capacity

    @property
    def n_plants(self):                 # getter: runs on tray.n_plants
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):          # setter: runs on tray.n_plants = value
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise ValueError(f"the tray only has {self.capacity} cells")
        self._n_plants = value

    @property
    def free_cells(self):               # computed from other attributes
        return self.capacity - self.n_plants

    def add_plants(self, n):
        self.n_plants += n              # also goes through the setter

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

## Exercise 2.1

`type(e).__name__` gives the name of the class of the exception `e`.

In [ ]:
tray = Tray("Pansy", 20)

try:
    Tray("Pansy", 30, capacity=24)
except Exception as e:
    print(type(e).__name__, "-", e)

try:
    tray.n_plants = -5
except Exception as e:
    print(type(e).__name__, "-", e)

try:
    tray.add_plants(100)
except Exception as e:
    print(type(e).__name__, "-", e)

try:
    tray.capacity = 100
except Exception as e:
    print(type(e).__name__, "-", e)

print(tray)   # the tray did not change

The first three raise `ValueError`, from the setter of `n_plants`, which runs in `__init__`, in `tray.n_plants = -5`, and in `add_plants`. The last one raises `AttributeError`, because the property `capacity` has no setter.

## Exercises 2.2 and 2.3

- 2.2: `from_csv_row` reads three values, and passes the capacity to `cls(...)`.
- 2.3: `species` becomes a property. Its setter runs in `__init__` (`self.species = species`), so invalid species are rejected when the tray is created. `isinstance(value, str)` checks that the value is a string.

In [ ]:
class Tray:
    CAPACITIES = (24, 50, 104)

    def __init__(self, species, n_plants=0, capacity=50):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity
        self.species = species
        self.n_plants = n_plants

    @classmethod
    def from_csv_row(cls, row):                                     # 2.2
        species, n_plants, capacity = row.split(",")
        return cls(species.strip(), int(n_plants), int(capacity))

    @property
    def species(self):                                              # 2.3
        return self._species

    @species.setter
    def species(self, value):                                       # 2.3
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError("species must be a non-empty string")
        self._species = value.strip()

    @property
    def capacity(self):
        return self._capacity

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise ValueError(f"the tray only has {self.capacity} cells")
        self._n_plants = value

    @property
    def free_cells(self):
        return self.capacity - self.n_plants

    def add_plants(self, n):
        self.n_plants += n

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

Class `Inventory` of Class 9 works with the new `Tray` without any change: it only uses `tray.species`, `tray.n_plants` and `print(tray)`, which still exist (`n_plants` and `species` are now properties, but they are used in the same way).

In [ ]:
nursery = Inventory("ISA nursery")
nursery.add_tray("A1", Tray.from_csv_row("Pansy,25,50"))
nursery.add_tray("A2", Tray.from_csv_row("Petunia,24,24"))
nursery.add_tray("B1", Tray("Begonia", capacity=104))

nursery.list_all_trays()
print(nursery.n_plants(), nursery.plants_by_species(), nursery.empty_trays())

In [ ]:
for species in ["", "   ", 42]:
    try:
        Tray(species, 10)
    except ValueError as e:
        print(repr(species), "-", e)

---
# Part 3 — When to use a class

## Exercise 3.1

The two classes become two functions, in a module `units.py`. In a script, you would create the file `units.py` in the editor. Here, the notebook writes the file, and then imports it.

In [ ]:
code = """def kg_to_tons(kg):
    return kg / 1000


def ha_to_m2(ha):
    return ha * 10000


def germination_rate(n_seeds, n_germinated):
    return n_germinated / n_seeds
"""
with open("units.py", "w") as file:
    file.write(code)

In [ ]:
import units

print(units.kg_to_tons(2500))
print(units.ha_to_m2(1.5))
print(units.germination_rate(100, 87))

Compare with the class versions: `Converter().kg_to_tons(2500)` and `GerminationRate(100, 87).compute()`. The functions are shorter, and they do the same thing.

## Exercise 3.2

| case | class or function? | why |
|---|---|---|
| (a) convert °F to °C | function | one computation, no data to keep: `(f - 32) * 5 / 9` |
| (b) greenhouse with sensors and alarms | class | several sensors, each with its own readings, which change over time; several operations (add a reading, compute the mean, check the alarm). A class `Sensor`, and a class `Greenhouse` that *has* sensors (composition) |
| (c) area of a field | function | one computation from two numbers |
| (d) seed bank | class | several seed lots, each with its own species, quantity and expiry date; lots are added, withdrawn and expire, and quantities must stay valid. A class `SeedLot`, and a class `SeedBank` that *has* lots, like `Inventory` and `Tray` |

---
# Part 4 — Inheritance, used sparingly, and composition

## Exercise 4.1

`CapacityError` is raised in the setter of `n_plants` when there are too many plants. A negative number of plants still raises `ValueError`.

In [ ]:
class CapacityError(ValueError):
    """Raised when a tray does not have room for the plants."""


class Tray:
    CAPACITIES = (24, 50, 104)

    def __init__(self, species, n_plants=0, capacity=50):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity
        self.species = species
        self.n_plants = n_plants

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants, capacity = row.split(",")
        return cls(species.strip(), int(n_plants), int(capacity))

    @property
    def species(self):
        return self._species

    @species.setter
    def species(self, value):
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError("species must be a non-empty string")
        self._species = value.strip()

    @property
    def capacity(self):
        return self._capacity

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise CapacityError(f"the tray only has {self.capacity} cells")   # 4.1
        self._n_plants = value

    @property
    def free_cells(self):
        return self.capacity - self.n_plants

    def add_plants(self, n):
        self.n_plants += n

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

In [ ]:
tray = Tray("Pansy", 20)

try:
    tray.add_plants(100)
except ValueError as e:
    print(type(e).__name__, isinstance(e, ValueError), isinstance(e, CapacityError))

try:
    tray.n_plants = -5
except ValueError as e:
    print(type(e).__name__, isinstance(e, ValueError), isinstance(e, CapacityError))

- `tray.add_plants(100)` raises a `CapacityError`, which *is a* `ValueError`: both `isinstance` return `True`, and `except ValueError` catches it.
- `tray.n_plants = -5` raises a `ValueError`, which is not a `CapacityError`.

`except CapacityError` catches only the first one:

In [ ]:
try:
    tray.n_plants = -5
except CapacityError:
    print("not enough room")
except ValueError:
    print("some other problem with the number of plants")

## 4.2 A child class with `super()`

In [ ]:
class SeedTray(Tray):
    def __init__(self, species, n_seeds=0, capacity=50, germination_rate=0.8):
        super().__init__(species, n_seeds, capacity)   # runs __init__ of Tray
        self.germination_rate = germination_rate

    def expected_plants(self):
        return round(self.n_plants * self.germination_rate)

    def __str__(self):                                  # overrides __str__ of Tray
        return f"{super().__str__()}, {self.expected_plants()} expected to germinate"


seeds = SeedTray("Basil", 50, germination_rate=0.7)
print(seeds)
print(seeds.free_cells, seeds.capacity)                # inherited from Tray
print(isinstance(seeds, SeedTray), isinstance(seeds, Tray))

## Exercise 4.4

`Inventory` works with seed trays without any change (composition and polymorphism): `print(tray)` in `list_all_trays` calls the `__str__` of `SeedTray` for the seed tray, and the `__str__` of `Tray` for the others.

In [ ]:
nursery = Inventory("ISA nursery")
nursery.add_tray("A1", Tray("Pansy", 25))
nursery.add_tray("A2", Tray("Pansy", 32))
nursery.add_tray("S1", SeedTray("Basil", 50, germination_rate=0.7))

nursery.list_all_trays()
print(nursery.plants_by_species())

Note that `plants_by_species` counts the 50 basil **seeds** as plants, since a `SeedTray` stores its seeds in `n_plants`. The code works, but is the result what the nursery wants? Deciding what a method should mean for each kind of object is part of designing classes.

---
# Part 5 — On your own

## Exercises 5.1 and 5.4

The final version of class `Tray`:

- 5.1: `__add__` returns a **new** tray, with the plants of both trays and the capacity of the larger one. If the plants do not fit, `CapacityError` is raised by the setter of `n_plants`, when the new tray is created.
- 5.4: the attributes `material`, `max_uses` (`None` means no limit) and `n_uses`. In `add_plants`, `self.n_uses` is increased **after** the plants are added, so that a failed `add_plants` does not count as a use.

In [ ]:
class Tray:
    CAPACITIES = (24, 50, 104)

    def __init__(self, species, n_plants=0, capacity=50, material="plastic", max_uses=None):
        if capacity not in Tray.CAPACITIES:
            raise ValueError(f"capacity must be one of {Tray.CAPACITIES}")
        self._capacity = capacity
        self.species = species
        self.n_plants = n_plants
        self.material = material        # 5.4
        self.max_uses = max_uses        # 5.4: None means no limit
        self.n_uses = 0                 # 5.4

    @classmethod
    def from_csv_row(cls, row):
        species, n_plants, capacity = row.split(",")
        return cls(species.strip(), int(n_plants), int(capacity))

    @property
    def species(self):
        return self._species

    @species.setter
    def species(self, value):
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError("species must be a non-empty string")
        self._species = value.strip()

    @property
    def capacity(self):
        return self._capacity

    @property
    def n_plants(self):
        return self._n_plants

    @n_plants.setter
    def n_plants(self, value):
        if value < 0:
            raise ValueError("n_plants cannot be negative")
        if value > self.capacity:
            raise CapacityError(f"the tray only has {self.capacity} cells")
        self._n_plants = value

    @property
    def free_cells(self):
        return self.capacity - self.n_plants

    def add_plants(self, n):
        if self.max_uses is not None and self.n_uses >= self.max_uses:     # 5.4
            raise ValueError(f"the tray was already used {self.max_uses} times")
        self.n_plants += n
        self.n_uses += 1                                                   # 5.4

    def remove_plants(self, n):
        self.n_plants -= n

    def __str__(self):
        return f"{self.n_plants} {self.species} plants ({self.free_cells} free cells)"

    def __add__(self, other):                                              # 5.1
        if self.species != other.species:
            raise ValueError("cannot merge trays of different species")
        capacity = max(self.capacity, other.capacity)
        return Tray(self.species, self.n_plants + other.n_plants, capacity)

In [ ]:
# 5.1
small = Tray("Pansy", 20, capacity=24)
large = Tray("Pansy", 30)
merged = small + large            # calls small.__add__(large)
print(merged)
print(small, "|", large)          # the original trays did not change

for a, b in [(Tray("Pansy"), Tray("Petunia")), (Tray("Pansy", 30), Tray("Pansy", 30))]:
    try:
        a + b
    except ValueError as e:
        print(type(e).__name__, "-", e)

In [ ]:
# 5.4
tray = Tray("Pansy", capacity=24, max_uses=2)
tray.add_plants(20)
tray.remove_plants(20)
tray.add_plants(24)
tray.remove_plants(24)
try:
    tray.add_plants(10)           # third use
except ValueError as e:
    print(e)
print(tray, "- used", tray.n_uses, "times")

unlimited = Tray("Pansy")          # max_uses=None
for _ in range(5):
    unlimited.add_plants(1)
print(unlimited, "- used", unlimited.n_uses, "times")

## Exercises 5.2 and 5.3

The final version of class `Inventory`, with the methods of Class 9 and the new methods `load_csv` (5.2), `free_cells` and `full_trays` (5.3).

In `load_csv`, `line.split(",", maxsplit=1)` splits each line only at the first comma: the first part is the tray name, and the rest (`Pansy,25,50`) is passed to `Tray.from_csv_row`. Empty lines are skipped.

In [ ]:
class Inventory:
    def __init__(self, name):
        self.name = name
        self.trays = {}

    def add_tray(self, tray_name, tray):
        self.trays[tray_name] = tray

    def get_tray(self, tray_name):
        return self.trays[tray_name]

    def remove_tray(self, tray_name):
        if tray_name not in self.trays:
            raise ValueError(f"there is no tray {tray_name}")
        del self.trays[tray_name]

    def list_all_trays(self):
        for tray_name, tray in self.trays.items():
            print(f"{tray_name}: {tray}")

    def n_plants(self):
        return sum(tray.n_plants for tray in self.trays.values())

    def plants_by_species(self):
        counts = {}
        for tray in self.trays.values():
            counts[tray.species] = counts.get(tray.species, 0) + tray.n_plants
        return counts

    def empty_trays(self):
        return [tray_name for tray_name, tray in self.trays.items() if tray.n_plants == 0]

    def total_value(self, prices):
        return sum(tray.n_plants * prices[tray.species] for tray in self.trays.values())

    def load_csv(self, path):                                               # 5.2
        with open(path) as file:
            for line in file:
                line = line.strip()
                if line == "":
                    continue
                tray_name, row = line.split(",", maxsplit=1)
                self.add_tray(tray_name, Tray.from_csv_row(row))

    def free_cells(self):                                                   # 5.3
        return sum(tray.free_cells for tray in self.trays.values())

    def full_trays(self):                                                   # 5.3
        return [tray_name for tray_name, tray in self.trays.items() if tray.free_cells == 0]

    def __str__(self):
        return f"{self.name}: {len(self.trays)} trays"

A small file to test `load_csv` (in the codespace, you can also create it in the editor):

In [ ]:
with open("trays.csv", "w") as file:
    file.write("A1,Pansy,25,50\n")
    file.write("A2,Pansy,50,50\n")
    file.write("B1,Petunia,24,24\n")
    file.write("C1,Begonia,0,104\n")

In [ ]:
nursery = Inventory("ISA nursery")
nursery.load_csv("trays.csv")
print(nursery)
nursery.list_all_trays()
print("free cells:", nursery.free_cells())
print("full trays:", nursery.full_trays())
print("empty trays:", nursery.empty_trays())